# Preprocessing: reef live coral cover change (`lcc_change`)

**Target:** `lcc_change` = live coral cover this survey − live coral cover at the previous survey (percentage points).

**Framing: forecast.** Features may only use information available *before* the survey whose change we predict:
- the previous survey's cover (`lcc_prev`) and its other measurements (fish, invertebrates, substrate, impacts), shifted back one survey
- sea-temperature stress (NOAA DHW / SST anomaly) for the survey year and the year before
- location

**Evaluation (done in `modelling.ipynb`):** rolling origin. For each year from 2018 to 2025, train on all earlier years and test on that year. This notebook keeps all 348 rows together.

Outputs go to `data/processed/`.

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)

DATA_PATH = Path('master_reef_tourism_dataset.csv')
OUT_DIR = Path('data/processed')
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = 'lcc_change'

GRP = ['grp_other', 'grp_available_substrate', 'grp_sand',
       'grp_disturbance_indicators', 'grp_pollution_indicators']
FISH = ['fish_butterflyfish', 'fish_snapper', 'fish_parrotfish',
        'fish_grouper', 'fish_sweetlips', 'fish_moray_eel']
INV = ['inv_diadema_urchin', 'inv_sea_cucumber', 'inv_giant_clam', 'inv_crown_of_thorns']
IMPACT = ['impact_anchor', 'impact_nets', 'impact_trash', 'impact_bleaching', 'impact_cot']
OTHER_SURVEY = ['grazer_ratio', 'cot_outbreak', 'n_highvalue_fish_absent', 'n_curio_inverts_absent']

df = pd.read_csv(DATA_PATH).sort_values(['island', 'survey_year']).reset_index(drop=True)
print(df.shape)
df.head()

## 1. Leakage audit

These identities hold in the data, so any column that contains the *current* survey's coral cover gives the target away:

| Column | Relationship |
|---|---|
| `live_coral_cover_pct` | target = this − `lcc_prev` |
| `lcc_change_rate` | target ÷ `years_since_prev` |
| `island_vs_region_pct` | this = current cover − `region_avg_lcc_pct` |
| `grp_*` (5 substrate groups) | live coral + the five groups = 100% of the survey points |
| `region_avg_lcc_pct` | same-year regional average, which likely includes this island's current cover |

The substrate groups are the least obvious: every survey point is assigned to exactly one category, so when coral goes down the other groups must go up by the same amount, whatever the cause.

In [ ]:
has_target = df[TARGET].notna()

# target = current cover - previous cover
err = (df['live_coral_cover_pct'] - df['lcc_prev'] - df[TARGET])[has_target].abs()
assert err.max() < 0.01, err.max()

# target rate = target / years between surveys
err = (df[TARGET] / df['years_since_prev'] - df['lcc_change_rate'])[has_target].abs()
assert err.max() < 0.01, err.max()

# live coral + five substrate groups = 100%
total = df['live_coral_cover_pct'] + df[GRP].sum(axis=1, min_count=len(GRP))
assert total.dropna().between(99.5, 100.5).all(), total.describe()

# island_vs_region = current cover - regional average
err = (df['live_coral_cover_pct'] - df['region_avg_lcc_pct'] - df['island_vs_region_pct']).abs()
assert err.max() < 0.01, err.max()

# Consequence: the target can be rebuilt without any model
rebuilt = 100 - df[GRP].sum(axis=1, min_count=len(GRP)) - df['lcc_prev']
ok = rebuilt.notna() & has_target
r = np.corrcoef(rebuilt[ok], df.loc[ok, TARGET])[0, 1]
print(f'100 - sum(grp_*) - lcc_prev vs {TARGET}: r = {r:.4f} on {ok.sum()} rows')

## 2. Clean `ecoregion`

Labels are inconsistent within islands:
- **"Sulu Sea"** (11 rows) is applied on and off to three Sabah islands (Penyu is split 5–5 with "North Borneo"), and it gets exactly the same `region_avg_lcc_pct` as "North Borneo" every year. The source treats them as one region, so it is merged into North Borneo.
- A couple of one-off mislabels remain (Lankayan 2013 "Malacca Strait", Sembilan 2013 "Sunda Shelf"). Each row gets its island's majority label.

Two islands have no label at all, but each was surveyed only once, so those rows have no target and are dropped in step 5.

In [ ]:
before = df['ecoregion'].copy()

# Same regional average every year -> same region in the source
same_avg = df[df['ecoregion'].isin(['Sulu Sea', 'North Borneo'])].groupby(['survey_year', 'ecoregion'])['region_avg_lcc_pct'].first().unstack()
assert same_avg.dropna().nunique(axis=1).eq(1).all()
df['ecoregion'] = df['ecoregion'].replace({'Sulu Sea': 'North Borneo'})

def island_majority(labels):
    counts = labels.value_counts()
    if counts.empty:
        return np.nan
    assert len(counts) == 1 or counts.iloc[0] > counts.iloc[1], 'tied ecoregion labels'
    return counts.index[0]

df['ecoregion'] = df['island'].map(df.groupby('island')['ecoregion'].agg(island_majority))

changed = ~((before == df['ecoregion']) | (before.isna() & df['ecoregion'].isna()))
print(f'{changed.sum()} rows relabelled; levels now: {df["ecoregion"].value_counts().to_dict()}')
print(f'still missing: {df.loc[df["ecoregion"].isna(), "island"].tolist()} (single-survey islands, no target)')
pd.DataFrame({'island': df['island'], 'state': df['state'], 'survey_year': df['survey_year'],
              'before': before, 'after': df['ecoregion']})[changed]

## 3. Feature engineering: previous-survey features

Done on the full table **before** dropping rows, because each island's first survey (which has no target) is still needed as the "previous survey" for its second one.

- **Substrate → share of non-coral space.** `share_k = grp_k / Σgrp × 100`. Shares always total 100% of the non-coral area, so they can't reveal the coral total. Available substrate is the reference level and is left out (the shares sum to 100).
- **Same-year survey measurements → previous survey's value** (`prev_*`): fish, invertebrates, impacts, crown-of-thorns outbreak, grazer ratio, absence counts.
- **`prev_lcc_change`** (the previous survey's change) and **`prev_island_vs_region_pct`**. The lagged regional average is not added: it equals `lcc_prev − prev_island_vs_region_pct`.

`shift(1)` within island = the previous survey, which is exactly how the dataset defines `lcc_prev` (checked below).

- **`n_eff_sites`** (not a feature; used for training weights): each change is the difference of two survey averages, so its survey noise depends on both site counts: `n_eff = 1 / (1/n_sites + 1/prev_n_sites)`. Missing `n_sites` are filled with the island's usual count, or the overall median if the island never records one.

In [ ]:
SHARE_NAMES = {'grp_other': 'other', 'grp_available_substrate': 'avail', 'grp_sand': 'sand',
               'grp_disturbance_indicators': 'disturb', 'grp_pollution_indicators': 'pollution'}
noncoral = df[GRP].sum(axis=1, min_count=len(GRP))
for col, name in SHARE_NAMES.items():
    df[f'share_{name}'] = df[col] / noncoral * 100

SHARES = [f'share_{n}' for n in SHARE_NAMES.values()]
LAG_COLS = ([s for s in SHARES if s != 'share_avail']
            + FISH + INV + IMPACT + OTHER_SURVEY
            + [TARGET, 'island_vs_region_pct'])

by_island = df.groupby('island')
df = pd.concat([df, by_island[LAG_COLS].shift(1).add_prefix('prev_')], axis=1)

# shift(1) lines up with the dataset's own lcc_prev
prev_cover = by_island['live_coral_cover_pct'].shift(1)
m = df['lcc_prev'].notna()
assert np.allclose(prev_cover[m], df.loc[m, 'lcc_prev'])

# Survey reliability behind each change (training weights only, never a feature)
n_filled = (df['n_sites'].fillna(df.groupby('island')['n_sites'].transform('median'))
            .fillna(df['n_sites'].median()))
df['n_eff_sites'] = 1 / (1 / n_filled + 1 / n_filled.groupby(df['island']).shift(1))

df[['island', 'survey_year', 'lcc_prev', 'prev_lcc_change', 'prev_share_disturb',
    'prev_share_pollution', 'prev_fish_parrotfish', 'prev_impact_trash']].head(8)

## 4. Drop columns

Every dropped column is listed with its reason. `island` stays as an ID (for the grouped folds) but is never a feature; `survey_year` stays as a split column and a numeric trend feature.

In [ ]:
DROP = {
    'leak: current coral cover or derived from it':
        ['live_coral_cover_pct', 'lcc_change_rate', 'island_vs_region_pct', 'region_avg_lcc_pct'] + GRP,
    'measured after the survey':
        ['dhw_max_jan_apr_next', 'ssta_mean_jan_apr_next'],
    'same-year survey measurements (previous-survey versions kept)':
        SHARES + FISH + INV + IMPACT + OTHER_SURVEY,
    'unused categoricals / extraction metadata':
        ['state', 'noaa_station_id', 'marine_park',
         'confidence', 'substrate6_source', 'fish_source', 'n_sites_source'],
}
drop_cols = [c for cols in DROP.values() for c in cols]
assert set(drop_cols) <= set(df.columns), set(drop_cols) - set(df.columns)

for reason, cols in DROP.items():
    print(f'{reason} ({len(cols)}): {", ".join(cols)}')

data = df.drop(columns=drop_cols)
print(f'\n{data.shape[1]} columns kept:', list(data.columns))

## 5. Drop rows without a target

Each island's first survey has no previous survey, so `lcc_change` is missing. Those rows are dropped rather than imputed: an imputed target would be made-up training labels.

In [ ]:
n_before = len(data)
data = data.dropna(subset=[TARGET]).reset_index(drop=True)
print(f'{n_before} -> {len(data)} rows ({n_before - len(data)} first surveys removed)')
data[TARGET].describe()

## 6. Evaluation design: no fixed test set

All 348 rows are kept together. The modelling notebook evaluates with a **rolling origin**: for each year t from 2018 to 2025, train on every survey before t and test on year t. So every forecast only uses the past, every year's shock (including the 2024–2025 drop) is tested once, and the later years also become training data for the forecasts after them. Settings are tuned inside each origin with island-grouped CV on the past years only.

Survey counts and the mean change per year:

In [ ]:
data.groupby('survey_year').agg(rows=('island', 'size'), islands=('island', 'nunique'),
                                mean_change=(TARGET, 'mean')).round(2).T

## 7. Imputation, scaling, encoding

| Group | Columns | Steps |
|---|---|---|
| Continuous | cover, time, location, heat stress, previous-survey shares/fish/invertebrates | median `SimpleImputer` → `StandardScaler` |
| Binary | previous-survey impact flags, crown-of-thorns outbreak | most-frequent `SimpleImputer` (kept as 0/1, not scaled) |
| Categorical | `ecoregion` | most-frequent `SimpleImputer` → `OneHotEncoder` (unseen categories ignored) |

The saved preprocessor is fitted on all 348 rows, and is meant only for the final model trained on all years. For evaluation, the modelling notebook refits a fresh copy (`sklearn.base.clone`) inside every origin and fold, so test years never influence the imputer or scaler.

In [ ]:
CATEGORICAL = ['ecoregion']
BINARY = [f'prev_{c}' for c in IMPACT + ['cot_outbreak']]
NON_FEATURES = ['island', TARGET, 'n_eff_sites']
assert data['n_eff_sites'].notna().all()
CONTINUOUS = [c for c in data.columns if c not in CATEGORICAL + BINARY + NON_FEATURES]
FEATURES = CONTINUOUS + BINARY + CATEGORICAL

print(f'{len(CONTINUOUS)} continuous: {CONTINUOUS}')
print(f'{len(BINARY)} binary: {BINARY}')
print(f'{len(CATEGORICAL)} categorical: {CATEGORICAL}')

preprocessor = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')),
                      ('scale', StandardScaler())]), CONTINUOUS),
    ('bin', SimpleImputer(strategy='most_frequent'), BINARY),
    ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                      ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), CATEGORICAL),
], verbose_feature_names_out=False).set_output(transform='pandas')

X = preprocessor.fit_transform(data[FEATURES])
y = data[TARGET]
print(f'\nX {X.shape}')

# missing values handled per column
data[FEATURES].isna().sum().loc[lambda s: s > 0].sort_values(ascending=False).to_frame('n_missing')

## 8. Checks

In [ ]:
LEAKY = {'live_coral_cover_pct', 'lcc_change_rate', 'island_vs_region_pct', 'region_avg_lcc_pct',
         'dhw_max_jan_apr_next', 'ssta_mean_jan_apr_next', TARGET, *GRP, *SHARES}
assert not LEAKY & set(FEATURES), LEAKY & set(FEATURES)
assert not X.isna().any().any()
assert np.allclose(X[CONTINUOUS].mean(), 0, atol=1e-8)
assert np.allclose(X[CONTINUOUS].std(ddof=0), 1, atol=1e-6)
assert set(np.unique(X[BINARY])) <= {0.0, 1.0}
assert len(X) == len(data)

# A leaky feature set would let even a plain linear fit reproduce the target (R² ≈ 1).
r2 = LinearRegression().fit(X, y).score(X, y)
print(f'In-sample linear R² on the processed features: {r2:.3f} (a leak would give ~1.000)')
print('All checks passed.')

## 9. Save

| File | Contents |
|---|---|
| `features_raw.csv` | All 348 rows: engineered features **before** imputation/scaling, plus `island`, `survey_year`, `lcc_change` and `n_eff_sites` (for training weights). Use this for evaluation, refitting the preprocessor inside each origin/fold. |
| `X.csv`, `y.csv` | Transformed features (preprocessor fitted on all rows) and target, same row order as `features_raw.csv`. |
| `preprocessor.joblib` | The `ColumnTransformer` fitted on all 348 rows, for the final model. `clone()` gives an unfitted copy. |

In [ ]:
data.to_csv(OUT_DIR / 'features_raw.csv', index=False)
X.to_csv(OUT_DIR / 'X.csv', index=False)
y.to_frame().to_csv(OUT_DIR / 'y.csv', index=False)
joblib.dump(preprocessor, OUT_DIR / 'preprocessor.joblib')

for f in sorted(OUT_DIR.iterdir()):
    print(f'{f.name:26s} {f.stat().st_size / 1024:7.1f} KB')

## Using these in modelling

Rolling-origin evaluation, refitting the preprocessor on the past years at each origin:

```python
import joblib, pandas as pd
from sklearn.base import clone
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import make_pipeline

raw = pd.read_csv('data/processed/features_raw.csv')
pre = joblib.load('data/processed/preprocessor.joblib')
features = list(pre.feature_names_in_)

for t in range(2018, 2026):
    past, now = raw[raw.survey_year < t], raw[raw.survey_year == t]
    model = make_pipeline(clone(pre), Ridge(alpha=100)).fit(past[features], past['lcc_change'])
    print(t, mean_absolute_error(now['lcc_change'], model.predict(now[features])))
```